# Homework 3 - MLlib

The purpose of this assignment is for you to use Spark MLlib for machine learning.

- **Please leave ample time for this homework, as it may take a while to complete**.
- We expect the steps of this assignment to be idempotent -- i.e., they are re-run safe and generate the same result if you run them again (except for the manual file uploading step, which requires our manual intervention).

#  Part A

### 1. Compare Spark ML with another machine learning framework of your choice (preferably one you are less familiar with).
- Describe the main characteristics of both frameworks.
- Identify at least two strengths and two weaknesses for each framework.
- Discuss how you would choose between the frameworks for at least two business scenarios.

If you use sources to support your answers, be sure to credit them.

Below is a comparison between Spark MLlib and Scikit-learn, addressing their main characteristics, strengths, weaknesses, and business scenario choices.

**Spark MLlib**

- **Main characteristics:** Distributed machine learning library built on Apache Spark, designed for scalability and integration with big data workflows.
- **Strengths:**
  - Scales efficiently to large datasets across clusters.
  - Integrates seamlessly with Spark's data processing and production pipelines.
- **Weaknesses:**
  - Fewer algorithms and less flexibility compared to Scikit-learn.
  - Visualization and interpretability tools are limited.
- **Business scenario examples:**
  - *Scenario 1:* Predicting customer churn for a telecom company with millions of records. MLlib is preferred due to its scalability and distributed processing.
  - *Scenario 2:* Real-time fraud detection in financial transactions, where integration with Spark streaming is required.

**Scikit-learn**

- **Main characteristics:** Popular Python library for machine learning, focused on single-machine processing, rich algorithm selection, and ease of use.
- **Strengths:**
  - Extensive selection of algorithms and preprocessing tools.
  - Mature, well-documented, and easy to use with strong community support.
- **Weaknesses:**
  - Limited to single-machine processing; not suitable for very large datasets.
  - Not designed for distributed computing or integration with big data pipelines.
- **Business scenario examples:**
  - *Scenario 1:* Building a prototype model for a small e-commerce dataset. Scikit-learn is preferred for its ease of use and rapid iteration.
  - *Scenario 2:* Academic research requiring advanced algorithms and interpretability.


In [ ]:
Comparison: Spark MLlib vs H2O

Spark MLlib

Main characteristics:
Spark MLlib is a distributed machine learning library built on Apache Spark. It is designed for large-scale data processing and machine learning tasks by distributing computations across clusters. It integrates closely with Spark components such as Spark SQL, DataFrames, and data pipelines, which makes it suitable for big data environments.

Strengths

Scalability: Can process very large datasets using distributed computing across multiple nodes.

Integration with big data systems: Works seamlessly with Spark tools such as Spark SQL, ETL pipelines, and streaming frameworks.

Pipeline support: Enables building scalable machine learning pipelines for preprocessing, feature engineering, and modeling.

Weaknesses

Limited algorithm variety: Provides fewer advanced algorithms compared to specialized machine learning frameworks.

Infrastructure requirements: Requires a Spark cluster, which can increase setup and operational complexity.

Business scenario examples

Customer churn prediction in telecommunications companies with millions of customer records stored in distributed systems.

Fraud detection in financial institutions where large volumes of transaction data must be processed in real time.

H2O

Main characteristics:
H2O is an open-source distributed machine learning platform designed for scalable machine learning and automated model development. It provides high-performance implementations of algorithms such as Gradient Boosting Machines, Random Forest, and Deep Learning, and includes AutoML capabilities that automatically train and compare multiple models.

Strengths

AutoML capability: Automatically trains multiple models and selects the best-performing one.

High-performance algorithms: Provides optimized implementations of advanced algorithms for strong predictive accuracy.

Scalable architecture: Supports distributed computing for handling large datasets.

Weaknesses

Less integration with data processing frameworks: Compared with Spark MLlib, it does not integrate as naturally with full big data ecosystems and ETL pipelines.

Additional system setup: Requires managing a separate H2O environment alongside existing data infrastructure.

Business scenario examples

Credit risk modeling in banking, where automated model selection can improve predictive accuracy.

Marketing campaign optimization, where companies use AutoML to identify the best-performing predictive model for customer targeting.


AI Acknowledgment: I used ChatGPT as a tool to clarify machine learning concepts, organize preliminary ideas, and refine the explanations written in this assignment.


# Part B. Analyze Titanic Dataset using Spark ML

This is a well-known dataset for machine learning. A description of the dataset can be found at the [kaggle website](https://www.kaggle.com/c/titanic/data). In the following, we apply the logistic regression model from Spark ML. The goal is to predict the survival of passagers on board titanic, and to use the pipeline and feature engineering tools of Spark ML. 

Though the data is small (and does not warrant the use of scalable Spark ML), we can use it to learn the process of building a feature engineering and/or ML pipeline using Spark ML. 

Throughout the analyses, we expect to 
- use **Spark SQL** or **Spark DataFrame APIs** to maintain scalability
- **maintain idempotency** (you can run the notebook twice without generating errors or changing the results)

![Data Dictionary](https://i.ibb.co/mhY89DG/titanic-datadictionary.png)

## Dataset Uploading

### 1. Download and load data
- Create a Unity Catalog Volume `homework3` under the schema `spark` of the catalog `msbabigdata` (recreate them if not exists)
- Download [`titanic.csv`](https://idsdl.csom.umn.edu/c/share/msba6330/titanic.csv)
- Then, load the data into the Unity Catalog Volume `homework3`.
- Create a Spark DataFrame `titanic` using the uploaded csv file.

In [ ]:


%sql
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.homework3;

In [ ]:
#Create a Spark DataFrame 
titanic = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("/Volumes/msbabigdata/spark/homework3/titanic.csv")
)

titanic.display()

PassengerId | Survived | Pclass | Name | Sex | Age | SibSp | Parch | Ticket | Fare | Cabin | Embarked
1 | 0 | 3 | Braund, Mr. Owen Harris | male | 22.0 | 1 | 0 | A/5 21171 | 7.25 | None | S
2 | 1 | 1 | Cumings, Mrs. John Bradley (Florence Briggs Thayer) | female | 38.0 | 1 | 0 | PC 17599 | 71.2833 | C85 | C
3 | 1 | 3 | Heikkinen, Miss. Laina | female | 26.0 | 0 | 0 | STON/O2. 3101282 | 7.925 | None | S
4 | 1 | 1 | Futrelle, Mrs. Jacques Heath (Lily May Peel) | female | 35.0 | 1 | 0 | 113803 | 53.1 | C123 | S
5 | 0 | 3 | Allen, Mr. William Henry | male | 35.0 | 0 | 0 | 373450 | 8.05 | None | S
6 | 0 | 3 | Moran, Mr. James | male | None | 0 | 0 | 330877 | 8.4583 | None | Q
7 | 0 | 1 | McCarthy, Mr. Timothy J | male | 54.0 | 0 | 0 | 17463 | 51.8625 | E46 | S
8 | 0 | 3 | Palsson, Master. Gosta Leonard | male | 2.0 | 3 | 1 | 349909 | 21.075 | None | S
9 | 1 | 3 | Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg) | female | 27.0 | 0 | 2 | 347742 | 11.1333 | None | S
10 | 1 | 2 | Nasser, Mrs.

### 2. Verify the DataFrame by

- Displaying the schema
- Displaying the first 5 rows of data
- Counting the number of rows

In [ ]:
# Display the schema
titanic.printSchema()



root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [ ]:
# Display first 5 rows
titanic.show(5)



+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+------

In [ ]:
# Count the number of rows
print("Number of rows:", titanic.count())

Number of rows: 891


### 3. Split the DataFrame into Training and Testing Sets

- Randomly split the DataFrame into `train` (70%) and `test` (30%), using 1234 as the random seed.
- Print the row count for each DataFrame.

In [ ]:
train, test = titanic.randomSplit([0.7, 0.3], seed=1234)

spark.createDataFrame([
    ("Training", train.count()),
    ("Testing", test.count())
], ["Dataset", "Rows"]).display()

Dataset | Rows
Training | 603
Testing | 288

### 4. Persist the `train` and `test` DataFrames as Spark tables in the database `titanic`

- Create the database `titanic` if it does not exist
- The table names will be `train` and `test`
- Keep in mind that all steps need to be idempotent.

In [ ]:
# Create database if it does not exist
spark.sql("CREATE DATABASE IF NOT EXISTS titanic")

# Save train and test as tables (idempotent)
train.write.mode("overwrite").saveAsTable("titanic.train")
test.write.mode("overwrite").saveAsTable("titanic.test")

# to verify
spark.sql("SHOW TABLES IN titanic").display()

database | tableName | isTemporary
titanic | test | False
titanic | train | False

### 6. Discuss the differences between the new and original DataFrames

- Discuss the differences between using the new DataFrames (`train_table` and `test_table`) and the original ones for subsequent analyses.
- Are there advantages to using the newer ones?

In [ ]:
Differences between the original DataFrames and the new persisted DataFrames
1. Storage
Original DataFrames (train and test): Exist only within the current Spark session unless explicitly saved.
New DataFrames / tables (train_table and test_table): Persisted as Spark tables in the titanic database and stored for future use.

2. Availability
Original DataFrames: Need to be recreated if the notebook session ends or the cluster restarts.
New persisted tables: Can be loaded again at any time without repeating the split step.

3. Reusability
Original DataFrames: Mainly useful within the same notebook run.
New persisted tables: Can be reused across multiple notebook cells, workflows, or future sessions.

4. Reproducibility
Original DataFrames: Depend on rerunning the code correctly to regenerate the same split.
New persisted tables: Preserve the exact train-test split once saved, making later analyses more consistent.


Advantages of using the newer persisted DataFrames / tables
1. Better reproducibility
The exact same training and testing sets are available for later analyses, ensuring consistency in results.

2. Greater convenience
There is no need to rerun the splitting process each time the notebook is opened or the cluster restarts.

3. Improved idempotency
Persisted tables make the workflow safer and more stable because the same saved data can be reused repeatedly.

4. Easier organization
Saving the data in the titanic database makes the workflow more structured and easier to manage.

5. Better collaboration and reuse
Other analyses or notebooks can directly access the saved tables without recreating them.

Conclusion - 
Yes, there are clear advantages of using teh newer persisted dataframes/tables. They are more reproducable, reusable nd convenient than the original temporary dataframes.

## Feature Engineering

The goal of this step is to perform feature engineering to prepare for ML models. We want to do it in a way that's repeatable—so that feature engineering steps can be codified in a pipeline and applied for both model training and inference.

We will focus on:
- converting data types
- selecting columns
- dealing with missing values
- handling text and categorical columns
- assembling the feature vector

### 7. Based on `train`, create a new DataFrame `train_cast` that:
- consists of only the features and label (`Survived`) columns that are plausible and will be used:
  - Survived
  - Pclass
  - Age
  - SibSp
  - Parch
  - Sex
  - Fare
  - Embarked
- converts all numerical columns into double type. Although Spark may automatically convert them to double when you assemble vectors, we use this as a chance to practice type conversion.

Then show the first 10 rows and verify the schema.

In [ ]:
from pyspark.sql.functions import col
train = spark.table("titanic.train")
train_cast = (
    train
    .select(
        col("Survived").cast("double"),
        col("Pclass").cast("double"),
        col("Age").cast("double"),
        col("SibSp").cast("double"),
        col("Parch").cast("double"),
        col("Sex"),
        col("Fare").cast("double"),
        col("Embarked")
    )
)

# Show first 10 rows
train_cast.show(10)

# Verify schema
train_cast.printSchema()

+--------+------+----+-----+-----+------+-------+--------+
|Survived|Pclass| Age|SibSp|Parch|   Sex|   Fare|Embarked|
+--------+------+----+-----+-----+------+-------+--------+
|     0.0|   3.0|22.0|  1.0|  0.0|  male|   7.25|       S|
|     1.0|   1.0|38.0|  1.0|  0.0|female|71.2833|       C|
|     1.0|   3.0|26.0|  0.0|  0.0|female|  7.925|       S|
|     1.0|   1.0|35.0|  1.0|  0.0|female|   53.1|       S|
|     0.0|   3.0|35.0|  0.0|  0.0|  male|   8.05|       S|
|     0.0|   3.0|NULL|  0.0|  0.0|  male| 8.4583|       Q|
|     0.0|   1.0|54.0|  0.0|  0.0|  male|51.8625|       S|
|     0.0|   3.0| 2.0|  3.0|  1.0|  male| 21.075|       S|
|     1.0|   3.0|27.0|  0.0|  2.0|female|11.1333|       S|
|     1.0|   3.0| 4.0|  1.0|  1.0|female|   16.7|       S|
+--------+------+----+-----+-----+------+-------+--------+
only showing top 10 rows
root
 |-- Survived: double (nullable = true)
 |-- Pclass: double (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: double (nullable = 

### 8. Handling Missing Ages

We notice that there are missing values in the `Age` and `Embarked` columns. We decide to use a mean replacement strategy for missing ages and a drop-rows strategy for missing `Embarked`. 

- First, use DataFrame.describe() to obtain the mean age.
- Then create a new DataFrame `train_missing` after you:
  - Eliminate the rows where `Embarked` is NULL.
  - Create a new `AgeFilled` column that is a version of `Age` with null values replaced by the mean age obtained in the previous step.
  - Create a new DOUBLE indicator column **AgeNA** to denote whether the `Age` value was missing (1.0 indicates missing, 0.0 indicates not missing).

Verify the new DataFrame by printing the first 10 rows and the count of rows.

In [ ]:
train_cast.describe("Age").show()

+-------+-----------------+
|summary|              Age|
+-------+-----------------+
|  count|              488|
|   mean|30.02340163934426|
| stddev|14.06413565095182|
|    min|             0.42|
|    max|             74.0|
+-------+-----------------+



In [ ]:
#mean age value
display(train_cast.selectExpr("avg(Age) as Mean_Age"))

Mean_Age
30.02340163934426

In [ ]:
#create train_missing
from pyspark.sql.functions import col, when

# get mean age
mean_age = train_cast.selectExpr("avg(Age)").first()[0]

train_missing = (
    train_cast
    .na.drop(subset=["Embarked"])  # remove rows with missing Embarked
    .withColumn("AgeNA", when(col("Age").isNull(), 1.0).otherwise(0.0))
    .na.fill({"Age": mean_age})  # fill missing Age values
    .withColumnRenamed("Age", "AgeFilled")
)

train_missing.show(10)
train_missing.count()

+--------+------+-----------------+-----+-----+------+-------+--------+-----+
|Survived|Pclass|        AgeFilled|SibSp|Parch|   Sex|   Fare|Embarked|AgeNA|
+--------+------+-----------------+-----+-----+------+-------+--------+-----+
|     0.0|   3.0|             22.0|  1.0|  0.0|  male|   7.25|       S|  0.0|
|     1.0|   1.0|             38.0|  1.0|  0.0|female|71.2833|       C|  0.0|
|     1.0|   3.0|             26.0|  0.0|  0.0|female|  7.925|       S|  0.0|
|     1.0|   1.0|             35.0|  1.0|  0.0|female|   53.1|       S|  0.0|
|     0.0|   3.0|             35.0|  0.0|  0.0|  male|   8.05|       S|  0.0|
|     0.0|   3.0|30.02340163934426|  0.0|  0.0|  male| 8.4583|       Q|  1.0|
|     0.0|   1.0|             54.0|  0.0|  0.0|  male|51.8625|       S|  0.0|
|     0.0|   3.0|              2.0|  3.0|  1.0|  male| 21.075|       S|  0.0|
|     1.0|   3.0|             27.0|  0.0|  2.0|female|11.1333|       S|  0.0|
|     1.0|   3.0|              4.0|  1.0|  1.0|female|   16.7|  

In [ ]:
#to verify
train_missing.show(10)

print("Row count:", train_missing.count())

+--------+------+-----------------+-----+-----+------+-------+--------+-----+
|Survived|Pclass|        AgeFilled|SibSp|Parch|   Sex|   Fare|Embarked|AgeNA|
+--------+------+-----------------+-----+-----+------+-------+--------+-----+
|     0.0|   3.0|             22.0|  1.0|  0.0|  male|   7.25|       S|  0.0|
|     1.0|   1.0|             38.0|  1.0|  0.0|female|71.2833|       C|  0.0|
|     1.0|   3.0|             26.0|  0.0|  0.0|female|  7.925|       S|  0.0|
|     1.0|   1.0|             35.0|  1.0|  0.0|female|   53.1|       S|  0.0|
|     0.0|   3.0|             35.0|  0.0|  0.0|  male|   8.05|       S|  0.0|
|     0.0|   3.0|30.02340163934426|  0.0|  0.0|  male| 8.4583|       Q|  1.0|
|     0.0|   1.0|             54.0|  0.0|  0.0|  male|51.8625|       S|  0.0|
|     0.0|   3.0|              2.0|  3.0|  1.0|  male| 21.075|       S|  0.0|
|     1.0|   3.0|             27.0|  0.0|  2.0|female|11.1333|       S|  0.0|
|     1.0|   3.0|              4.0|  1.0|  1.0|female|   16.7|  


String columns cannot be used directly with some models such as LogisticRegression. Neither can categorical columns. Our strategy is to convert string and categorical variables into a series of binary dummies. Fortunately, these steps are already provided as part of pyspark.ml.features.

To convert indexed categorical values into a vector of binary indicators, we can leverage `OneHotEncoder`.

Finally, we need to assemble a vector column with all the numerical features. This is achieved by `VectorAssembler`.

Click on each below to familiarize yourself with them if you need further information.
- [StringIndexer](https://spark.apache.org/docs/latest/api/python/pyspark.ml.html#pyspark.ml.feature.StringIndexer): converts string labels into numerical labels (0, 1, ...), ordered by label frequencies
- [OneHotEncoder](https://spark.apache.org/docs/latest/api/python/pyspark.ml.html#pyspark.ml.feature.OneHotEncoder): maps a column of category indices to a column of binary vectors (the least frequent one will be dropped by default).
- [VectorAssembler](https://spark.apache.org/docs/latest/api/python/pyspark.ml.html#pyspark.ml.feature.VectorAssembler): merges multiple columns into one vector column needed for most algorithms


### 9. Index and Encode string/categorical columns

String columns cannot be used directly with some models such as LogisticRegression, even after we convert them into numeric categorical columns (0, 1, 2, ...). Our strategy is to first index string columns using StringIndexer (so you get values such as 0, 1, 2, ...), then use OneHotEncoder to convert categorical values into binary dummies.

- Create a StringIndexer `si` for indexing the two string features `Sex` and `Embarked` (the resultant columns should have a postfix of `_idx`).
- Then create a OneHotEncoder `ohe` for the indexed string columns (the resulting columns should have a postfix of `_ohe`).

Test them by applying them on the previous DataFrame and display the first 10 rows.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# create stringindexer
si = StringIndexer(
    inputCols=["Sex", "Embarked"],
    outputCols=["Sex_idx", "Embarked_idx"]
)

#create OneHotCoder
ohe = OneHotEncoder(
    inputCols=["Sex_idx", "Embarked_idx"],
    outputCols=["Sex_ohe", "Embarked_ohe"]
)

In [ ]:
# Apply StringIndexer
train_indexed = si.fit(train_missing).transform(train_missing)

# Apply OneHotEncoder
train_encoded = ohe.fit(train_indexed).transform(train_indexed)

In [ ]:
train_encoded.show(10)


+--------+------+-----------------+-----+-----+------+-------+--------+-----+-------+------------+-------------+-------------+
|Survived|Pclass|        AgeFilled|SibSp|Parch|   Sex|   Fare|Embarked|AgeNA|Sex_idx|Embarked_idx|      Sex_ohe| Embarked_ohe|
+--------+------+-----------------+-----+-----+------+-------+--------+-----+-------+------------+-------------+-------------+
|     0.0|   3.0|             22.0|  1.0|  0.0|  male|   7.25|       S|  0.0|    0.0|         0.0|(1,[0],[1.0])|(2,[0],[1.0])|
|     1.0|   1.0|             38.0|  1.0|  0.0|female|71.2833|       C|  0.0|    1.0|         1.0|    (1,[],[])|(2,[1],[1.0])|
|     1.0|   3.0|             26.0|  0.0|  0.0|female|  7.925|       S|  0.0|    1.0|         0.0|    (1,[],[])|(2,[0],[1.0])|
|     1.0|   1.0|             35.0|  1.0|  0.0|female|   53.1|       S|  0.0|    1.0|         0.0|    (1,[],[])|(2,[0],[1.0])|
|     0.0|   3.0|             35.0|  0.0|  0.0|  male|   8.05|       S|  0.0|    0.0|         0.0|(1,[0],[1.0])

### 10. Assemble feature columns into a feature vector (for use with the pipeline)

Assemble all features into one vector column; we shall call the new column `features`.

- For the `Age` column, include both `AgeFilled` and `AgeNA`.
- For `Sex` and `Embarked`, include only the encoded versions produced by OneHotEncoder.

Verify this step by applying it to the previous DataFrame and display the first 10 rows.

In [ ]:
from pyspark.ml.feature import VectorAssembler

va = VectorAssembler(
    inputCols=[
        "Pclass",
        "AgeFilled",
        "AgeNA",
        "SibSp",
        "Parch",
        "Fare",
        "Sex_ohe",
        "Embarked_ohe"
    ],
    outputCol="features"
)

train_features = va.transform(train_encoded)

train_features.select("features").limit(10).display()

features
{"type":"1","size":null,"indices":null,"values":["3.0","22.0","0.0","1.0","0.0","7.25","1.0","1.0","0.0"]}
{"type":"1","size":null,"indices":null,"values":["1.0","38.0","0.0","1.0","0.0","71.2833","0.0","0.0","1.0"]}
{"type":"0","size":"9","indices":["0","1","5","7"],"values":["3.0","26.0","7.925","1.0"]}
{"type":"1","size":null,"indices":null,"values":["1.0","35.0","0.0","1.0","0.0","53.1","0.0","1.0","0.0"]}
{"type":"1","size":null,"indices":null,"values":["3.0","35.0","0.0","0.0","0.0","8.05","1.0","1.0","0.0"]}
{"type":"1","size":null,"indices":null,"values":["3.0","30.02340163934426","1.0","0.0","0.0","8.4583","1.0","0.0","0.0"]}
{"type":"1","size":null,"indices":null,"values":["1.0","54.0","0.0","0.0","0.0","51.8625","1.0","1.0","0.0"]}
{"type":"1","size":null,"indices":null,"values":["3.0","2.0","0.0","3.0","1.0","21.075","1.0","1.0","0.0"]}
{"type":"1","size":null,"indices":null,"values":["3.0","27.0","0.0","0.0","2.0","11.1333","0.0","1.0","0.0"]}
{"type":"1","size":n

## Models and ML Pipelines

We will try a few different models including Logistic Regression and Random Forest, and also tune the parameters of the Random Forest model.

### 11. Create the LogisticRegression model (for use with the pipeline)
- The documentation for LogisticRegression from pyspark.ml can be [found here](https://spark.apache.org/docs/latest/api/python/pyspark.ml.html#pyspark.ml.classification.LogisticRegression).
- Create the LogisticRegression estimator `lr` to predict `Survived` using our assembled feature vector.
- Verify it by applying it to the previous DataFrame and displaying the first 10 rows of the result.

In [ ]:
#create logistic regression model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    labelCol="Survived",
    featuresCol="features"
)

In [ ]:
#fit the model on the previous dataframe
lr_model = lr.fit(train_features)

In [ ]:
# apply the model and verify the results
lr_predictions = lr_model.transform(train_features)

lr_predictions.select("Survived", "features", "prediction", "probability").limit(10).display()

Survived | features | prediction | probability
0.0 | {"type":"1","size":null,"indices":null,"values":["3.0","22.0","0.0","1.0","0.0","7.25","1.0","1.0","0.0"]} | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.9352302209989443","0.06476977900105574"]}
1.0 | {"type":"1","size":null,"indices":null,"values":["1.0","38.0","0.0","1.0","0.0","71.2833","0.0","0.0","1.0"]} | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.052267971033784874","0.9477320289662151"]}
1.0 | {"type":"0","size":"9","indices":["0","1","5","7"],"values":["3.0","26.0","7.925","1.0"]} | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.3724904595999013","0.6275095404000988"]}
1.0 | {"type":"1","size":null,"indices":null,"values":["1.0","35.0","0.0","1.0","0.0","53.1","0.0","1.0","0.0"]} | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.07356540068812323","0.9264345993118768"]}
0.0 | {"type":"1","size":null,"indices":null,"values":["3.0","35.0","0.0","0.0","0.0","8.05","1.0","1.0","0

### 12. Define the Pipeline 

Put together the feature engineering and ML pipeline `pl`.

- The first step is casting and column selection.
- The last step is logistic regression.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, OneHotEncoder, VectorAssembler, Imputer
from pyspark.ml.classification import LogisticRegression

# Step 1: select columns, cast numerics, drop missing Embarked, create AgeNA
cast_sql = SQLTransformer(statement="""
SELECT
    CAST(Survived AS DOUBLE) AS Survived,
    CAST(Pclass AS DOUBLE) AS Pclass,
    CAST(Age AS DOUBLE) AS Age,
    CASE WHEN Age IS NULL THEN 1.0 ELSE 0.0 END AS AgeNA,
    CAST(SibSp AS DOUBLE) AS SibSp,
    CAST(Parch AS DOUBLE) AS Parch,
    Sex,
    CAST(Fare AS DOUBLE) AS Fare,
    Embarked
FROM __THIS__
WHERE Embarked IS NOT NULL
""")

# Step 2: fill missing Age with mean -> AgeFilled
imputer = Imputer(
    inputCols=["Age"],
    outputCols=["AgeFilled"]
).setStrategy("mean")

# Step 3: index string columns
si = StringIndexer(
    inputCols=["Sex", "Embarked"],
    outputCols=["Sex_idx", "Embarked_idx"]
)

# Step 4: one-hot encode indexed columns
ohe = OneHotEncoder(
    inputCols=["Sex_idx", "Embarked_idx"],
    outputCols=["Sex_ohe", "Embarked_ohe"]
)

# Step 5: assemble features
va = VectorAssembler(
    inputCols=[
        "Pclass",
        "AgeFilled",
        "AgeNA",
        "SibSp",
        "Parch",
        "Fare",
        "Sex_ohe",
        "Embarked_ohe"
    ],
    outputCol="features"
)

# Step 6: logistic regression
lr = LogisticRegression(
    labelCol="Survived",
    featuresCol="features"
)

# Full pipeline
pl = Pipeline(stages=[cast_sql, imputer, si, ohe, va, lr])

### 13. Train the pipeline on the training set and use the trained pipeline on the test set

- Fit the pipeline on the training set `train_table` and save the result as `pl_model`.
- Use `pl_model` to obtain predictions on the test set `test_table`, saving the resulting DataFrame as `results`.
- Display the first 10 rows of the result.

In [ ]:
# Load persisted tables if needed
train_table = spark.table("titanic.train")
test_table = spark.table("titanic.test")

# Train the pipeline
pl_model = pl.fit(train_table)

# Get predictions on test set
results = pl_model.transform(test_table)

# Display first 10 rows
results.limit(10).display()

Survived | Pclass | Age | AgeNA | SibSp | Parch | Sex | Fare | Embarked | AgeFilled | Sex_idx | Embarked_idx | Sex_ohe | Embarked_ohe | features | rawPrediction | probability | prediction
1.0 | 2.0 | 14.0 | 0.0 | 1.0 | 0.0 | female | 30.0708 | C | 14.0 | 1.0 | 1.0 | {"type":"0","size":"1","indices":[],"values":[]} | {"type":"0","size":"2","indices":["1"],"values":["1.0"]} | {"type":"1","size":null,"indices":null,"values":["2.0","14.0","0.0","1.0","0.0","30.0708","0.0","0.0","1.0"]} | {"type":"1","size":null,"indices":null,"values":["-2.625254785577171","2.625254785577171"]} | {"type":"1","size":null,"indices":null,"values":["0.06753064545831941","0.9324693545416806"]} | 1.0
1.0 | 1.0 | 58.0 | 0.0 | 0.0 | 0.0 | female | 26.55 | S | 58.0 | 1.0 | 0.0 | {"type":"0","size":"1","indices":[],"values":[]} | {"type":"0","size":"2","indices":["0"],"values":["1.0"]} | {"type":"0","size":"9","indices":["0","1","5","7"],"values":["1.0","58.0","26.55","1.0"]} | {"type":"1","size":null,"indices":null

### 14. Evaluate model performance

- Define a function `evaluate` that calculates and prints the area under the ROC curve (AUC), F1 score, and accuracy metrics for the provided DataFrame.  
- Use the function to evaluate the model performance on the test data.

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

def evaluate(df):
    auc = BinaryClassificationEvaluator(
        labelCol="Survived",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    ).evaluate(df)

    f1 = MulticlassClassificationEvaluator(
        labelCol="Survived",
        predictionCol="prediction",
        metricName="f1"
    ).evaluate(df)

    accuracy = MulticlassClassificationEvaluator(
        labelCol="Survived",
        predictionCol="prediction",
        metricName="accuracy"
    ).evaluate(df)

    return spark.createDataFrame(
        [("AUC", auc), ("F1 Score", f1), ("Accuracy", accuracy)],
        ["Metric", "Value"]
    )

In [ ]:
evaluate(results).display()

Metric | Value
AUC | 0.7970404984423684
F1 Score | 0.7373399632178889
Accuracy | 0.7386759581881533

### 15. Fit a Random Forest model, obtain predictions, and report the performance metrics

- Create the random forest estimator `rf`
- Create the pipeline `rf_pipeline` with the Random Forest at the end.
- Train the pipeline and save the result as `rf_pipeline_model`
- Use the pipeline model to do inference on `test_table`
- Evaluate the performance using `evaluate()`

In [ ]:
#create random forest estimator
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="Survived",
    featuresCol="features",
    seed=1234
)

In [ ]:
#create the pipeline with the random forest at the end
rf_pipeline = Pipeline(stages=[cast_sql, imputer, si, ohe, va, rf])

In [ ]:
# train the pipeline and save the results
train_table = spark.table("titanic.train")
test_table = spark.table("titanic.test")

rf_pipeline_model = rf_pipeline.fit(train_table)

In [ ]:
#use the pipeline model
rf_results = rf_pipeline_model.transform(test_table)

rf_results.select("Survived", "prediction", "probability").limit(10).display()
# Evaluate
evaluate(rf_results).display()

Survived | prediction | probability
1.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.14975518649526484","0.8502448135047352"]}
1.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.19345941768831867","0.8065405823116814"]}
0.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.8078307622247785","0.19216923777522155"]}
1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.837058798825306","0.16294120117469402"]}
0.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.37852488469485923","0.6214751153051408"]}
0.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.9138724778383553","0.08612752216164461"]}
1.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.05797717651549332","0.9420228234845067"]}
1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.8724967246327826","0.12750327536721737"]}
0.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.22594479347840896","0.7740552065215911"]}
0.0 | 0.0 |

### 16. Tune the Random Forest Model

RandomForest has a few parameters that can be further tuned. We are considering the following:

- maxDepth: 5, 7
- numTrees: 20, 30

Use CrossValidator (3 folds) to find the best parameter combination, using `areaUnderROC` as the metric.

- Obtain the trained pipeline model `cv_model`.
- Apply the trained pipeline to the test set and report the three performance metrics.

Note: For cross-validation, Databricks requires an environment variable `SPARKML_TEMP_DFS_PATH` (for setting the path for intermediary results). You can set the environment variable by:

```python
import os
# Set the environment variable for SparkML caching
# Set the path to your homework volume
os.environ['SPARKML_TEMP_DFS_PATH'] = path
```

In [ ]:
import os
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Set SparkML temp path
path = "/Volumes/msbabigdata/spark/homework3/"
os.environ["SPARKML_TEMP_DFS_PATH"] = path

# Load train and test tables
train_table = spark.table("titanic.train")
test_table = spark.table("titanic.test")

# Random Forest estimator
rf = RandomForestClassifier(
    labelCol="Survived",
    featuresCol="features",
    seed=1234
)

# Pipeline with RF at the end
rf_pipeline = Pipeline(stages=[cast_sql, imputer, si, ohe, va, rf])

# Parameter grid
paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 7])
    .addGrid(rf.numTrees, [20, 30])
    .build()
)

# Evaluator for CV
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="Survived",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Cross-validator
cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=auc_evaluator,
    numFolds=3,
    seed=1234
)

# Train tuned model
cv_model = cv.fit(train_table)

# Apply tuned model on test set
cv_results = cv_model.transform(test_table)

# Show first 10 rows
cv_results.select("Survived", "prediction", "probability").limit(10).display()

# Evaluate tuned model
evaluate(cv_results).display()

Survived | prediction | probability
1.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.14975518649526484","0.8502448135047352"]}
1.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.19345941768831867","0.8065405823116814"]}
0.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.8078307622247785","0.19216923777522155"]}
1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.837058798825306","0.16294120117469402"]}
0.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.37852488469485923","0.6214751153051408"]}
0.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.9138724778383553","0.08612752216164461"]}
1.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.05797717651549332","0.9420228234845067"]}
1.0 | 0.0 | {"type":"1","size":null,"indices":null,"values":["0.8724967246327826","0.12750327536721737"]}
0.0 | 1.0 | {"type":"1","size":null,"indices":null,"values":["0.22594479347840896","0.7740552065215911"]}
0.0 | 0.0 |

### 17. What parameters were selected by the CrossValidator?

**Tip**: 
- The CrossValidator model has a `bestModel` that stores the chosen model.
- Since our chosen model is a pipeline, you need to get the stage of the pipeline that corresponds to the RandomForest model using `.stages[x]`.

Type your answer: 
- How many trees are in the best model, and what is the maximum depth of these trees?

In [ ]:
#Since the best model is a PipelineModel, the Random Forest model is the last stage of the pipeline, 
#so we access it with .stages[-1].

cv_model = cv.fit(train_table)
best_rf = cv_model.bestModel.stages[-1]

spark.createDataFrame([
    ("Number of Trees", best_rf.getNumTrees),
    ("Max Depth", best_rf.getOrDefault("maxDepth"))
], ["Parameter", "Value"]).display()

Parameter | Value
Number of Trees | 20
Max Depth | 5

In [ ]:
Number of trees in the best model: 20
Maximum depth: 5

### 18. Persist the trained model for later use

You have decided to save the trained logistic regression model as well as the best Random Forest model for later use.
- Save them in the homework volume under the folders `lr_pipeline_model` and `rf_pipeline_best_model`, respectively.
- To remain idempotent, use `[model].write().overwrite().save(path)` to overwrite existing saved models.

Then, test the saved model:
- Load the `rf_pipeline_best_model` using `PipelineModel.read().load([path])`.
- Use the reloaded model to do inference on the `test_table` again.
- Evaluate the model performance to see if you get the same result.

In [ ]:
lr_path = "/Volumes/msbabigdata/spark/homework3/lr_pipeline_model"
rf_path = "/Volumes/msbabigdata/spark/homework3/rf_pipeline_best_model"

In [ ]:
pl_model.write().overwrite().save(lr_path)

cv_model.bestModel.write().overwrite().save(rf_path)

In [ ]:
from pyspark.ml import PipelineModel

rf_pipeline_best_model = PipelineModel.read().load(rf_path)

In [ ]:
rf_results_reload = rf_pipeline_best_model.transform(test_table)

rf_results_reload.limit(10).display()

Survived | Pclass | Age | AgeNA | SibSp | Parch | Sex | Fare | Embarked | AgeFilled | Sex_idx | Embarked_idx | Sex_ohe | Embarked_ohe | features | rawPrediction | probability | prediction
1.0 | 2.0 | 14.0 | 0.0 | 1.0 | 0.0 | female | 30.0708 | C | 14.0 | 1.0 | 1.0 | {"type":"0","size":"1","indices":[],"values":[]} | {"type":"0","size":"2","indices":["1"],"values":["1.0"]} | {"type":"1","size":null,"indices":null,"values":["2.0","14.0","0.0","1.0","0.0","30.0708","0.0","0.0","1.0"]} | {"type":"1","size":null,"indices":null,"values":["2.9951037299052965","17.0048962700947"]} | {"type":"1","size":null,"indices":null,"values":["0.14975518649526484","0.8502448135047352"]} | 1.0
1.0 | 1.0 | 58.0 | 0.0 | 0.0 | 0.0 | female | 26.55 | S | 58.0 | 1.0 | 0.0 | {"type":"0","size":"1","indices":[],"values":[]} | {"type":"0","size":"2","indices":["0"],"values":["1.0"]} | {"type":"0","size":"9","indices":["0","1","5","7"],"values":["1.0","58.0","26.55","1.0"]} | {"type":"1","size":null,"indices":null,

In [ ]:
#evaluate
evaluate(rf_results_reload).display()

Metric | Value
AUC | 0.8311266874350985
F1 Score | 0.8136523189278264
Accuracy | 0.818815331010453

### 19. Clean Up

Clean up the volumes, folders, tables, and database generated for the homework.

In [ ]:
dbutils.fs.rm("/Volumes/msbabigdata/spark/homework3/lr_pipeline_model", True)
dbutils.fs.rm("/Volumes/msbabigdata/spark/homework3/rf_pipeline_best_model", True)

In [ ]:
spark.sql("DROP TABLE IF EXISTS titanic.train")
spark.sql("DROP TABLE IF EXISTS titanic.test")
spark.sql("DROP DATABASE IF EXISTS titanic CASCADE")

In [ ]:
%sql
DROP VOLUME IF EXISTS msbabigdata.spark.homework3;

In [ ]:
#to verify
spark.sql("SHOW DATABASES").display()

databaseName
default
dualcore
information_schema
msbabigdata

In [ ]:
#to verify
spark.sql("SHOW VOLUMES IN msbabigdata.spark").display()

database | volume_name
spark | lab8_2
spark | lab8_3

This is the end of the homework